# Qizil Vino Sifati Bashorati (Wine Quality Prediction)
Ushbu daftarda tasodifiy o'rmon (Random Forest) va qaror daraxti (Decision Tree) algoritmlari yordamida vino sifatini tasniflash (klassifikatsiya) masalasi tahlil qilingan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# 1. Ma'lumotlarni yuklash va dastlabki tozalash
URL = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
df = pd.read_csv(URL).drop_duplicates()

# Sifat ko'rsatkichi 6 va undan yuqori bo'lgan vinolarni yaxshi (1), qolganlarini yomon (0) deb belgilaymiz
df['good'] = (df['quality'] >= 6).astype(int)

# Model uchun kiruvchi belgilar (X) va maqsadli o'zgaruvchi (y)
X = df.drop(columns=['quality', 'good'])
y = df['good']

# Ma'lumotlarni o'qitish (train) va test (sinov) to'plamlariga ajratish
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Ma'lumotlar o'lchami: {df.shape}")
display(df.head())

In [ ]:
# 2. Boshlang'ich Random Forest modelini o'qitish va natijalarni baholash
rf_initial = RandomForestClassifier(random_state=42)
rf_initial.fit(X_train, y_train)

y_pred_initial = rf_initial.predict(X_test)
print("Dastlabki Random Forest Test Aniqligi (Accuracy):", round(rf_initial.score(X_test, y_test), 4))
print("\nKlassifikatsiya Hisoboti (Classification Report):")
print(classification_report(y_test, y_pred_initial))

In [ ]:
# 3. Daraxtlar sonining (n_estimators) model aniqligiga ta'sirini tekshirish
print("Har xil n_estimators qiymatlari uchun test to'plamidagi aniqlik:")
for n in [1, 10, 100]:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    test_score = m.score(X_test, y_test)
    print(f"n_estimators: {n:3d} -> Test Accuracy: {test_score:.4f}")

In [ ]:
# 4. Giperparametrlarni GridSearchCV orqali eng optimal variantga sozlash
param_grid = {
    'max_features': ['sqrt', 0.5, None],
    'max_depth': [3, 5, 10, None]
}

rf_base = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Eng yaxshi topilgan parametrlar:", grid_search.best_params_)
print("Eng yaxshi Kross-Validatsiya (CV) aniqligi:", round(grid_search.best_score_, 4))
print("Optimal modelning Test to'plamidagi aniqligi:", round(best_model.score(X_test, y_test), 4))

In [ ]:
# 5. Optimal model asosida eng muhim 3 ta belgini (Feature Importance) aniqlash
imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top_3_features = imp.head(3)

print("Random Forest modeliga ko'ra qaror qabul qilishda eng muhim 3 ta belgi:")
for i, (col, val) in enumerate(top_3_features.items(), 1):
    print(f"{i}. {col}: {val:.4f} ({val*100:.2f}%)")

In [ ]:
# 6. Kross-validatsiya (5-fold) orqali Decision Tree va Random Forest modellarini taqqoslash va chalkashlik matritsasini chizish
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dt = DecisionTreeClassifier(random_state=42)

# Kross-validatsiya yordamida har bir model uchun bashoratlarni yig'ish
y_pred_cv_dt = cross_val_predict(dt, X, y, cv=cv)
y_pred_cv_rf = cross_val_predict(best_model, X, y, cv=cv)

# Chalkashlik matritsalarini hisoblash
cm_cv_dt = confusion_matrix(y, y_pred_cv_dt)
cm_cv_rf = confusion_matrix(y, y_pred_cv_rf)

# Vizual grafik chizish
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix=cm_cv_dt, display_labels=['Yomon', 'Yaxshi']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title("Decision Tree (CV-5) Chalkashlik Matritsasi")

ConfusionMatrixDisplay(confusion_matrix=cm_cv_rf, display_labels=['Yomon', 'Yaxshi']).plot(ax=axes[1], cmap='Greens')
axes[1].set_title("Random Forest (CV-5) Chalkashlik Matritsasi")

plt.tight_layout()
plt.show()

In [ ]:
# Ensemble - ko'p modellikning birgalikdagi qarori
# n_estimators - o'rmondagi daraxtlar soni
# feature importance - belgining bashoratdagi hissasi

# random forest da overfitting bo'lmaydi, unga scalling ham kerak emas.
# uni ishlash prinsipi ensemble ga asoslanadi, ensemble esa voting-ga asoslanadi


In [2]:
import pandas as pd
URL = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
df = pd.read_csv(URL).drop_duplicates()
df.shape

(1359, 12)

In [4]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000
mean,8.310596,0.529478,0.272333,2.523400,0.088124,15.893304,46.825975,0.996709,3.309787,0.658705,10.432315,5.623252
std,1.736990,0.183031,0.195537,1.352314,0.049377,10.447270,33.408946,0.001869,0.155036,0.170667,1.082065,0.823578
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996700,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.430000,2.600000,0.091000,21.000000,63.000000,0.997820,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [5]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv(URL).drop_duplicates()
df['good'] = (df['quality'] >= 6).astype(int)
X = df.drop(columns=['quality', 'good'])
y = df['good']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
print(round(rf.score(X_test, y_test), 4))

0.75


In [8]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# o'rmondagi har bir daraxtni alohida sanaymiz
acc = [(t.predict(X_test.values) == y_test.values).mean()
      for t in rf.estimators_]
acc = np.array(acc)

print(round(acc.min(), 4))
print(round(acc.mean(), 4))
print(round(acc.max(), 4))
print(round(rf.score(X_test, y_test), 4))

0.6103
0.6602
0.7206
0.75


In [18]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, rf.predict(X_test))
print(cm)

[[128   0]
 [  0 144]]


In [22]:
from sklearn.metrics import classification_report

print(classification_report(y_test, rf.predict(X_test)))
X = df.drop(columns=['quality', 'good'])
y = df['good']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


rf = RandomForestClassifier(random_state=42)
dt = DecisionTreeClassifier(random_state=42)

rf_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
dt_scores = cross_val_score(dt, X, y, cv=5, scoring='accuracy')

print("Random Forest:")
print("Natijalar:", rf_scores.round(4))
print("O'rtacha:", round(rf_scores.mean(), 4))

print("\nDecision Tree:")
print("Natijalar:", dt_scores.round(4))
print("O'rtacha:", round(dt_scores.mean(), 4))

              precision    recall  f1-score   support

           0       0.72      0.77      0.74       128
           1       0.78      0.74      0.76       144

    accuracy                           0.75       272
   macro avg       0.75      0.75      0.75       272
weighted avg       0.75      0.75      0.75       272

Confusion Matrix:
[[ 98  30]
 [ 38 106]]

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.77      0.74       128
           1       0.78      0.74      0.76       144

    accuracy                           0.75       272
   macro avg       0.75      0.75      0.75       272
weighted avg       0.75      0.75      0.75       272

Random Forest:
Natijalar: [0.6691 0.6875 0.7904 0.7206 0.7491]
O'rtacha: 0.7233

Decision Tree:
Natijalar: [0.625  0.6544 0.6434 0.6765 0.6421]
O'rtacha: 0.6483


In [23]:
for k in [3, 5, 10, None]:
  m = RandomForestClassifier(max_depth=k, random_state=42)
  m.fit(X_train, y_train)
  print(k,
        round(m.score(X_train, y_train), 4),
        round(m.score(X_test, y_test), 4))

3 0.77 0.7537
5 0.8123 0.7647
10 0.9834 0.7574
None 1.0 0.75


In [26]:
for n in [1, 10, 100]:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    test_score = m.score(X_test, y_test)
    print(f"n_estimators: {n:3d}, Test accuracy: {test_score:.4f}")

n_estimators:   1, Test accuracy: 0.6985
n_estimators:  10, Test accuracy: 0.7574
n_estimators: 100, Test accuracy: 0.7500


In [30]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Parametrlarni solishtirish
max_features_values = ['sqrt', 0.5, None]
max_depth_values = [3, 5, 10, None]

results = []

for f in max_features_values:
    for d in max_depth_values:
        model = RandomForestClassifier(
            max_features=f,
            max_depth=d,
            random_state=42
        )

        model.fit(X_train, y_train)

        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)

        results.append({
            'max_features': str(f),
            'max_depth': str(d),
            'train': round(train_score, 4),
            'test': round(test_score, 4),
            'farq': round(train_score - test_score, 4)
        })

# Jadvalni chiqarish
results_df = pd.DataFrame(results)
print("Barcha kombinatsiyalar:")
print(results_df.to_string(index=False))

# 2. GridSearchCV orqali eng yaxshi parametrlarni topish
param_grid = {
    'max_features': ['sqrt', 0.5, None],
    'max_depth': [3, 5, 10, None]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nEng yaxshi parametrlar:")
print(grid.best_params_)

print("\nEng yaxshi CV accuracy:")
print(round(grid.best_score_, 4))

# 3. Eng yaxshi modelning train va test natijalari
best_model = grid.best_estimator_

train_acc = best_model.score(X_train, y_train)
test_acc = best_model.score(X_test, y_test)

print("\nEng yaxshi model natijalari:")
print("Train:", round(train_acc, 4))
print("Test:", round(test_acc, 4))
print("Farq:", round(train_acc - test_acc, 4))


Barcha kombinatsiyalar:
max_features max_depth  train   test   farq
        sqrt         3 0.7700 0.7537 0.0163
        sqrt         5 0.8123 0.7647 0.0476
        sqrt        10 0.9834 0.7574 0.2261
        sqrt      None 1.0000 0.7500 0.2500
         0.5         3 0.7654 0.7279 0.0375
         0.5         5 0.8178 0.7463 0.0715
         0.5        10 0.9908 0.7610 0.2298
         0.5      None 1.0000 0.7574 0.2426
        None         3 0.7654 0.7243 0.0411
        None         5 0.8280 0.7316 0.0963
        None        10 0.9908 0.7537 0.2371
        None      None 1.0000 0.7500 0.2500

Eng yaxshi parametrlar:
{'max_depth': 5, 'max_features': None}

Eng yaxshi CV accuracy:
0.7553

Eng yaxshi model natijalari:
Train: 0.828
Test: 0.7316
Farq: 0.0963


In [33]:
# Eng yaxshi modelimiz (best_model) yordamida feature importance-ni chiqaramiz
imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(3).round(4))

alcohol             0.4731
sulphates           0.1326
volatile acidity    0.1024
dtype: float64


In [37]:
top_3_features = imp.head(3)

print("Random Forest modeliga ko'ra eng muhim 3 ta belgi:")
for i, (col, val) in enumerate(top_3_features.items(), 1):
    print(f"{i}. {col}: {val:.4f} ({val*100:.2f}%)")

Random Forest modeliga ko'ra eng muhim 3 ta belgi:
1. alcohol: 0.4731 (47.31%)
2. sulphates: 0.1326 (13.26%)
3. volatile acidity: 0.1024 (10.24%)


In [41]:
# I have no special talent. I am only passionately curious!
print("I have no special talent. I am only passionately curious! by Umid Azamqulov")
# It's all about priority!
print("It's all about priority! by Umis Azamqulov")

I have no special talent. I am only passionately curious! by Umid Azamqulov
It's all about priority! by Umis Azamqulov
